# GPT2 ARCHITECTURE
In this jupyter notebook, we will draft simple implementation of GPT2 architecture using PyTorch.
We will follow the sections
- GPT backbone architecture
- Layer Normalization
- GELU activation function
- Feed Forward Network
- Shortcut connection
- Transformer block
- Final GPT2 architecture

In [27]:
# configuration of GPT2 124M model
model_config = {
    "vocab_size": 50257,
    "context_length": 1024,
    "embedding_dim": 768,
    "num_heads": 12,
    "num_layers": 12,
    "dropout": 0.1,
    "qkv_bias": False,
}

In [ ]:
# place holder for GPT2 architecture implementation
import torch
import torch.nn as nn

    
class DummyLayerNorm(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.epsilon = 1e-5
        self.scale = nn.Parameter(torch.ones(embedding_dim))
        self.shift = nn.Parameter(torch.zeros(embedding_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(variance + self.epsilon)
        x = self.scale * normalized + self.shift
        return x
    
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(config['embedding_dim'], 4 * config['embedding_dim']),
            nn.GELU(),
            nn.Linear(4 * config['embedding_dim'], config['embedding_dim'])
        )
    def forward(self, x):
        return self.layers(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_lengh, dropout):
        super().__init__()
        print("initializing multi head attention with d_in:", d_in, "d_out:", d_out, "num_heads:", num_heads, "context_length:", context_lengh, "dropout:", dropout)
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_size = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.Dropout = nn.Dropout(dropout)
        self.OutProj = nn.Linear(d_out, d_out) # output projection to combine heads
        self.register_buffer('mask', torch.tril(torch.ones(context_lengh, context_lengh), diagonal=0)) # precompute mask for max context length of 1000

    def forward(self,x):
        print("input shape:", x.shape)
        batch_size, num_tokens, d_in = x.shape
        querys = self.W_query(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        keys = self.W_key(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        values = self.W_value(x) # shape: (batch_size, num_tokens, d_out = num_heads * head_size)
        # print("querys shape:", querys.shape)
        # print("keys shape:", keys.shape)
        # print("values shape:", values.shape)
       
        querys = querys.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)
        keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)
        values = values.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1,2) # shape: (batch_size, num_heads, num_tokens, head_size)

        attn_scores = querys @ keys.transpose(-2,-1) # (batch_size, num_heads, num_tokens, head_size) @ (batch_size, num_heads, head_size, num_tokens) --> (batch_size, num_heads, num_tokens, num_tokens)
        # print("before masked: ", attn_scores)
        masked_attn_scores = attn_scores.masked_fill(self.mask[:num_tokens, :num_tokens] == 0, float('-inf')) # (batch_size, num_heads, num_tokens, num_tokens)
        # print("mask: ", self.mask)
        # print("after masked: ", masked_attn_scores)
        attn_weights = torch.softmax(masked_attn_scores / (keys.shape[-1] ** 0.5), dim=-1) # (batch_size, num_heads, num_tokens, num_tokens)
        attn_weights = self.Dropout(attn_weights) # (batch_size, num_heads, num_tokens, num_tokens)
        context_vector = attn_weights @ values # (batch_size, num_heads, num_tokens, num_tokens) @ (batch_size, num_heads, num_tokens, head_size) --> (batch_size, num_heads, num_tokens, head_size)

        context_vector = context_vector.transpose(1,2) # (batch_size, num_tokens, num_heads, head_size)
        # print("context: ", context_vector)
        #combine heads
        context_vector = context_vector.contiguous().view(batch_size, num_tokens, self.num_heads * self.head_size) # (batch_size, num_tokens, d_out)
        # print("combined context: ", context_vector)

        context_vector = self.OutProj(context_vector) # (batch_size, num_tokens, d_out)
        return context_vector

    
class DummyTransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = MultiHeadAttention(d_in=config['embedding_dim'],
            d_out=config['embedding_dim'],
            context_lengh=config['context_length'],
            num_heads=config['num_heads'],
            dropout=config['dropout'])
        self.feed_forward = FeedForward(config)
        self.norm1 = DummyLayerNorm(config['embedding_dim'])
        self.norm2 = DummyLayerNorm(config['embedding_dim'])
        self.dropout = nn.Dropout(config['dropout'])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.feed_forward(x)
        x = self.dropout(x)
        x = x + shortcut
        return x

class DummyGPT2(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding = nn.Embedding(config['vocab_size'], config['embedding_dim'])
        self.position_embedding = nn.Embedding(config['context_length'], config['embedding_dim'])
        self.dropout_embedding = nn.Dropout(config['dropout'])
        self.transformer_blocks = nn.Sequential(*[DummyTransformerBlock(config) for _ in range(config['num_layers'])])
        self.final_norm = DummyLayerNorm(config['embedding_dim'])
        self.output_head = nn.Linear(config['embedding_dim'], config['vocab_size'], bias=False)

    def forward(self, in_idx):
        batch_size, seq_length = in_idx.shape
        token_embeddings = self.token_embedding(in_idx)
        position_embeddings = self.position_embedding(torch.arange(seq_length))
        x = self.dropout_embedding(token_embeddings + position_embeddings)
        x = self.transformer_blocks(x)
        x = self.final_norm(x)
        logits = self.output_head(x)
        return logits



In [37]:
torch.manual_seed(123)
x = torch.rand(2,4,768)
block = DummyTransformerBlock(model_config)
output = block(x)
print("input shape: ", x.shape)
print("output shape: ", output.shape)



initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
input shape: torch.Size([2, 4, 768])
input shape:  torch.Size([2, 4, 768])
output shape:  torch.Size([2, 4, 768])


In [41]:
# prepare input
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
text1 = "Every effort moves you"
text2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(text1)))
batch.append(torch.tensor(tokenizer.encode(text2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [39]:
torch.manual_seed(123)
x = torch.rand(2,4,768)
block = DummyTransformerBlock(model_config)
output = block(x)

print("output shape: ", output.shape)



initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
input shape: torch.Size([2, 4, 768])
input shape:  torch.Size([2, 4, 768])
output shape:  torch.Size([2, 4, 768])


In [40]:
torch.manual_seed(123)
model = DummyGPT2(model_config)
logits = model(batch)
print("INput batch shape: ", batch.shape)
print("Output logits shape: ", logits.shape)

initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head attention with d_in: 768 d_out: 768 num_heads: 12 context_length: 1024 dropout: 0.1
initializing multi head atte

In [51]:
def generate_text_simple(model, idx, max_new_token, context_size):
    for _ in range(max_new_token):
        idx_con = idx[:,-context_size:]
        print("idx: ", idx)
        print("idx_con: ", idx_con)
        with torch.no_grad():
            logits = model(idx_con)
        
        logits = logits[:, -1, :]
        prob = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(prob, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim= 1)
    return idx

start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded: ", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded tensor: ", encoded_tensor)

encoded:  [15496, 11, 314, 716]
encoded tensor:  tensor([[15496,    11,   314,   716]])


In [58]:
model.eval()
out = generate_text_simple(model=model, idx=encoded_tensor, max_new_token=6, context_size=model_config['context_length'])

print("OUT: ", out)
decoded = tokenizer.decode(out.squeeze(0).tolist())
print("decoded text: ", decoded)

idx:  tensor([[15496,    11,   314,   716]])
idx_con:  tensor([[15496,    11,   314,   716]])
input shape: torch.Size([1, 4, 768])
input shape: torch.Size([1, 4, 768])
input shape: torch.Size([1, 4, 768])
input shape: torch.Size([1, 4, 768])
input shape: torch.Size([1, 4, 768])
input shape: torch.Size([1, 4, 768])
input shape: torch.Size([1, 4, 768])
input shape: torch.Size([1, 4, 768])
input shape: torch.Size([1, 4, 768])
input shape: torch.Size([1, 4, 768])
input shape: torch.Size([1, 4, 768])
input shape: torch.Size([1, 4, 768])
idx:  tensor([[15496,    11,   314,   716, 27018]])
idx_con:  tensor([[15496,    11,   314,   716, 27018]])
input shape: torch.Size([1, 5, 768])
input shape: torch.Size([1, 5, 768])
input shape: torch.Size([1, 5, 768])
input shape: torch.Size([1, 5, 768])
input shape: torch.Size([1, 5, 768])
input shape: torch.Size([1, 5, 768])
input shape: torch.Size([1, 5, 768])
input shape: torch.Size([1, 5, 768])
input shape: torch.Size([1, 5, 768])
input shape: torch.Si